In [ ]:
import warnings
warnings.filterwarnings("ignore")
                        
import altair as alt
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils

credentials, _ = google.auth.default()

In [ ]:
sofi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times  
)

In [ ]:
sofi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times
)

# Do this separately, because we need stop's pt geom
arrivals_by_event_type = D2.aggregate_by_event_type(sofi_stop_arrivals)

## ideas
Contactless payments, before and after GIS map.
before event, more traffic
over time, how has that changed. more dynamic kind of chart, video?

heatmap, regression charts
* across dates, how it had changed, so it's little cubes, date on x-axis, routes on y-axis
* event on Saturday, compare to other saturdays (6 months or some period vs current Saturday with event)
* buffer ring analysis, near event, what changed 0-500meters, are there more trips, vs 500-1_000 meters, buffer to stop area, see changes to the stop. a buffer-ring analysis, splitting stops into distance bands from the stadium (say 0 to 500 meters, 500 meters to 1 kilometer, 1 to 2 kilometers), would show whether the service boost decays the further you get from the venue.

agency vs agency, specific agency that adds routes during event
mode related analysis, is it easier for bus routes to be added? or train frequencies?
date relative to event, google chart, event date is 0, and other dates are -1, 1, 2, 

bubble map, similar to ring map, but more suited for stops, can show change to bubble map for stops getting near stadiums

## Heatmap

In [ ]:
# heatmap
alt.Chart(sofi_trips).mark_rect().encode(
    y='route_name:O',
    x='service_date:T',
    color='sum(n_trips):Q'
)

In [ ]:
for i in arrivals_by_event_type.schedule_name.unique():
    one_chart = arrivals_by_operator_chart = alt.Chart(arrivals_by_event_type[arrivals_by_event_type.schedule_name==i]).mark_rect().encode(
        y='stop_name:O',
        x='event_day:O',
        color='sum(daily_arrivals):Q',
        tooltip=["sum(daily_arrivals)"]
    ).properties(title = i).interactive()
        
    display(one_chart)

## Map of Routes for all other feeds
* map of routes (how to compare event vs non-event)?
   * aggregate daily trips on event days (weekday + weekend), aggregate daily trips on non-event days (weekday + weekend)
   * show the difference? 
* can we get stops that show up on routes as a layer too?
* see a chart that filters for that route, show weekday + weekend event vs non-event trips?
* can be combined with World Cup feed

In [ ]:
trips_by_event = sofi_trips.groupby(
    ["schedule_name", "event_day", "day_type", "route_name", "route_type"]
).agg({
    "n_trips": "sum",
    "service_date": "nunique"
}).reset_index().rename(columns = {
    "service_date": "n_days"
})

trips_by_event = trips_by_event.assign(
    daily_trips = trips_by_event.n_trips.divide(trips_by_event.n_days).round(1)
)

In [ ]:
trips_by_event_wide = D2.make_wide(
    trips_by_event,
    group_cols = ["schedule_name", "day_type", "route_name"],
    metric_cols = ["daily_trips"]
)

In [ ]:
alt.Chart(trips_by_event).mark_rect().encode(
    y='route_name:O',
    x='event_day:O',
    color='sum(daily_trips):Q'
)

In [ ]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")
alt.Chart(trips_by_event).mark_bar().encode(
    x="daily_trips",
    y=alt.Y("event_day", title = ""),
    row=alt.Row("day_type:N", title = ""),
    color=alt.Color("schedule_name:N"), 
    opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(selection).transform_filter(selection)

In [ ]:
alt.Chart(trips_by_event).mark_bar().encode(
    x="daily_trips",
    y=alt.Y("event_day", title = ""),
    column=alt.Column("day_type:N", title = ""),
    row=alt.Row("schedule_name", title=""),
    color=alt.Color("schedule_name:N"), 
).properties(width=150, height=50)

In [ ]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")
alt.Chart(trips_by_event_wide).mark_bar().encode(
    x="change_daily_trips",
    y="route_name:N",
    column=alt.Column("day_type:N", title=""),
    color=alt.Color("schedule_name:N"), 
    opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(selection)